# YOFC G.654.E — 최신 웹 솔버 v0.2.1 재검증 (2026-11-03)

**재실행 엔진:** 동일하게 공개된 JavaScript `solver.mjs`의 `solve()`, `dispersion()`, `dispersionDiagnostics()` 함수. 기존 SciPy radial FEM과 수치적으로 교차 점검합니다.

**새 진단값:** S₁₅₅₀ = **0.059089**, 영분산 파장 λ₀ = **1275.457 nm**, S₀ = **0.093979** ps/(nm²·km). *S₀는 λ₀에서의 분산 기울기이며 YOFC의 S₁₅₅₀ 상한과 비교하는 수치가 아닙니다.*

![영분산 파장과 두 기울기 비교](assets/dispersion_slopes.svg)

[최신 웹 솔버 실행 JSON](./web_solver_rerun.json) · [HTML 시각화 보고서](./VISUAL_REPORT.html)

---

# 기존 SciPy FEM 대조 검증 / 그림으로 이해하기

> **검증 상태:** YOFC 사양에 맞춘 Step-index **가상 모델**의 수치 검증 · 실물 광섬유 검증/COMSOL 실행 아님

![FEM 결과 카드](assets/dashboard.svg)

**이 노트북에서 확인할 것:** ① 코어와 빛이 퍼지는 영역은 어떻게 다른가? ② 굴절률은 어디서 바뀌는가? ③ MFD·Aeff·D는 무엇을 뜻하고 얼마인가? ④ 파장을 바꾸면 결과가 어떻게 달라지는가? ⑤ 메시를 바꿔도 같은 결과가 나오는가?

- **YOFC 제공 사양:** MFD 11.4–12.2 µm, 대표 11.8 µm; Aeff 대표 110 µm²; D(1550 nm) ≤22 ps/(nm·km); D slope ≤0.070 ps/(nm²·km).
- **입력 가정:** a=5.4 µm, Δn=0.004, R(FEM 경계)=40 µm, λ=1550 nm, n₂=2.2×10⁻²⁰ m²/W, radial elements 750.
- **주의:** a와 Δn은 제품의 실제 도핑/굴절률 프로파일이 아니라 MFD와 Aeff에 맞춰 조정한 예시 입력입니다.

[자세한 한글 시각화 보고서](./VALIDATION_REPORT.md)

In [1]:
"""Research prototype v0.1: rotationally symmetric solid-silica fibers.

Scalar weak-guidance radial P1 finite element mode solver for LP01.
Materials are silica baseline Sellmeier plus *constant* delta_n offsets;
this is an illustrative index model, not a verified Ge/F doped Sellmeier law.
This solver does NOT model HCF/NANF, noncircular MCF, birefringence, loss, bends,
or fiber manufacturing variation. All transverse dimensions are in micrometres.
"""
from __future__ import annotations
from dataclasses import dataclass, replace, asdict
from typing import Sequence
import json
import numpy as np
from scipy.special import j0,j1,k0 as K0,k1 as K1
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from numpy.polynomial.legendre import leggauss

C0=299792458.0
_SELL_B=(0.6961663,0.4079426,0.8974794)
_SELL_C=(0.0684043**2,0.1162414**2,9.896161**2)

@dataclass(frozen=True)
class Model:
    profile: str = 'step'                  # step, graded, trench, multilayer
    wavelength_nm: float = 1550.0
    core_radius_um: float = 5.5
    delta_n_core: float = 0.0034785793846155
    outer_radius_um: float = 40.0       # computational, not physical cladding radius
    graded_alpha: float = 2.0
    trench_start_um: float = 7.0
    trench_end_um: float = 10.0
    trench_delta_n: float = -0.001
    # custom shells are (outer_radius_um, absolute delta_n relative to silica baseline).
    # First one starts at r=0; last listed shell may end before outer_radius_um.
    shells: tuple[tuple[float,float],...] = ((5.0,0.004), (7.0,0.0), (9.0,-0.0007))
    n2_m2_per_W: float = 2.2e-20
    def validate(self):
        if self.profile not in ('step','graded','trench','multilayer'):
            raise ValueError('profile must be step, graded, trench, or multilayer')
        for name in ('wavelength_nm','core_radius_um','outer_radius_um','graded_alpha','n2_m2_per_W'):
            x=float(getattr(self,name))
            if not np.isfinite(x) or x<=0: raise ValueError(f'{name} must be positive/finite')
        if not 250 < self.wavelength_nm < 3600: raise ValueError('wavelength outside silica Sellmeier domain')
        if self.outer_radius_um<=2.2*self.core_radius_um: raise ValueError('outer radius must exceed 2.2 x core radius')
        for name in ('delta_n_core','trench_delta_n'):
            if not np.isfinite(getattr(self,name)): raise ValueError(name)
        if self.profile=='trench' and not (self.core_radius_um < self.trench_start_um < self.trench_end_um < self.outer_radius_um):
            raise ValueError('trench radii must be core < start < end < outer')
        if self.profile=='multilayer':
            if not self.shells: raise ValueError('shells required')
            rr=np.array([x[0] for x in self.shells]); nn=np.array([x[1] for x in self.shells]);
            if np.any(~np.isfinite(rr)) or np.any(~np.isfinite(nn)) or np.any(np.diff(rr)<=0) or rr[0]<=0 or rr[-1]>=self.outer_radius_um:
                raise ValueError('shell boundaries must increase and lie below outer radius')
        return self

def silica_index(wavelength_um):
    w=np.asarray(wavelength_um,dtype=float)
    if np.any((w<=0.21)|(w>=3.7)): raise ValueError('silica Sellmeier only 0.21–3.7 µm')
    return np.sqrt(1+sum(b*w*w/(w*w-c) for b,c in zip(_SELL_B,_SELL_C)))

def boundaries(m:Model):
    m.validate()
    if m.profile=='trench': marks=[m.core_radius_um,m.trench_start_um,m.trench_end_um]
    elif m.profile=='multilayer': marks=[r for r,d in m.shells]
    else: marks=[m.core_radius_um]
    return np.r_[0.0,marks,m.outer_radius_um]

def delta_n_r(m:Model,r):
    r=np.asarray(r,dtype=float)
    if m.profile=='step': return np.where(r<m.core_radius_um,m.delta_n_core,0.0)
    if m.profile=='graded': return np.where(r<m.core_radius_um,m.delta_n_core*np.maximum(1-(r/m.core_radius_um)**m.graded_alpha,0),0.0)
    if m.profile=='trench':
        return np.where(r<m.core_radius_um,m.delta_n_core,
                        np.where((r>=m.trench_start_um)&(r<m.trench_end_um),m.trench_delta_n,0.0))
    out=np.zeros_like(r)
    left=0.0
    for right,dn in m.shells:
        out=np.where((r>=left)&(r<right),dn,out)
        left=right
    return out

def index_r(m:Model,r):
    return silica_index(m.wavelength_nm/1000)+delta_n_r(m,r)

def radial_grid(m:Model,elements=750):
    if not isinstance(elements,(int,np.integer)) or elements<80:raise ValueError('elements>=80')
    br=boundaries(m); intervals=np.diff(br)
    base=np.maximum(12,np.round(elements*intervals/np.sum(intervals)).astype(int))
    # allocate more near interfaces by maintaining at least 12 nodes per ring
    out=[np.linspace(br[i],br[i+1],int(base[i])+1)[:-1] for i in range(len(base))]
    return np.r_[np.concatenate(out),br[-1]]

def solve_lp01(m:Model,elements=750):
    m.validate(); r=radial_grid(m,elements)
    nodes=len(r); n_el=nodes-1
    t0,ww=leggauss(5); t=(t0+1)/2; wq=ww/2
    h=np.diff(r); rq=r[:-1,None]+h[:,None]*t
    weights=h[:,None]*wq
    basis=np.stack((1-t,t),axis=1)
    # quadratic-form angular integration has 2pi on both sides, cancels for eigenvalue.
    mass=np.einsum('eq,qi,qj->eij',weights*rq,basis,basis)
    deriv=(np.diff(r**2)/(2*h*h))[:,None,None]*np.array([[1.,-1.],[-1.,1.]])
    nvals=index_r(m,rq)
    n_mass=np.einsum('eq,eq,qi,qj->eij',weights*rq,nvals**2,basis,basis)
    k0=2*np.pi/(m.wavelength_nm/1000)
    mats=(k0*k0*n_mass-deriv,mass)
    edge=np.stack((np.arange(n_el),np.arange(1,nodes)),axis=1)
    ii=np.broadcast_to(edge[:,:,None],(n_el,2,2)).ravel()
    jj=np.broadcast_to(edge[:,None,:],(n_el,2,2)).ravel()
    A,B=[coo_matrix((v.ravel(),(ii,jj)),shape=(nodes,nodes)).tocsr()[:-1,:-1] for v in mats]
    n_max=float(np.max(index_r(m,rq)))
    eigen,ev=eigsh(A,k=1,M=B,sigma=(k0*n_max)**2+0.05,which='LM',tol=1e-10,v0=np.ones(nodes-1))
    beta2=float(eigen[0]); psi=np.r_[ev[:,0],0.0]
    if psi[0]<0:psi=-psi
    if beta2<=0: raise RuntimeError('nonpositive beta²')
    neff=float(np.sqrt(beta2)/k0)
    rel_res=float(np.linalg.norm(A@ev[:,0]-beta2*(B@ev[:,0]))/np.linalg.norm(beta2*(B@ev[:,0])))
    I=(psi[edge]@basis.T)**2
    wtr=weights*rq
    P=float(2*np.pi*np.sum(wtr*I)); Q=float(2*np.pi*np.sum(wtr*I**2));
    radial_moment=float(2*np.pi*np.sum(wtr*I*rq**2)/P)
    slope=np.diff(psi)/h
    nrm_grad=float(np.sum(weights*rq*slope[:,None]**2))
    nrm_psi=float(np.sum(weights*rq*I))
    mfd_pii=float(2*np.sqrt(2*nrm_psi/nrm_grad))
    aeff=P**2/Q
    tail_power=float(2*np.pi*np.sum(wtr[rq[:,0]>0.8*m.outer_radius_um]*I[rq[:,0]>0.8*m.outer_radius_um])/P)
    return {'model':m,'neff':neff,'Aeff_um2':aeff,'MFD_2nd_um':2*np.sqrt(2*radial_moment),'MFD_PII_um':mfd_pii,
            'gamma_Wkm':2*np.pi/(m.wavelength_nm*1e-9)*m.n2_m2_per_W/(aeff*1e-12)*1000,
            'residual':rel_res,'tail_power':tail_power,'guided_proxy':bool(neff>float(silica_index(m.wavelength_nm/1000))),
            'r_um':r,'field':psi/np.sqrt(P),'elements':n_el}

def bessel_step_oracle(m:Model):
    if m.profile!='step':raise ValueError('Bessel oracle only valid for step-index')
    k0=2*np.pi/(m.wavelength_nm/1000)
    ncl=float(silica_index(m.wavelength_nm/1000)); nco=ncl+m.delta_n_core
    if nco<=ncl:raise ValueError('index-guided core required')
    V=k0*m.core_radius_um*np.sqrt(nco*nco-ncl*ncl)
    if not (0<V<2.40482555769):raise ValueError('LP01 reference restricted to V<2.405')
    def fn(u):
        w=np.sqrt(V*V-u*u)
        return u*j1(u)/j0(u)-w*K1(w)/K0(w)
    u=brentq(fn,max(1e-8,V*1e-6),V*(1-1e-10),xtol=3e-13)
    neff=float(np.sqrt(nco*nco-(u/(k0*m.core_radius_um))**2))
    return {'neff':neff,'V':V}

def spectral_neff(m:Model,lambda_nm,elements=750):
    return solve_lp01(replace(m,wavelength_nm=float(lambda_nm)),elements)['neff']

def chromatic_dispersion(m:Model,elements=750,h_nm=10.0):
    # D ps/(nm.km) = -(lambda/c) d2 n_eff/d(lambda_um)2 * 1e12
    # All eigenvalue samples computed on identical radial mesh (wavelength-independent geometry).
    w=m.wavelength_nm/1000; h=h_nm/1000
    n=np.array([spectral_neff(m,(w+j*h)*1000,elements) for j in (-2,-1,0,1,2)])
    d2=(-n[4]+16*n[3]-30*n[2]+16*n[1]-n[0])/(12*h*h)
    return float(-w/C0*1e12*d2)

def serialize(sol):
    return {k:(asdict(v) if isinstance(v,Model) else float(v) if isinstance(v,(np.floating,np.integer)) else v)
            for k,v in sol.items() if k not in ('r_um','field')}

PRESETS={
 'G.654.E-like (illustrative)':dict(profile='step',core_radius_um=5.5,delta_n_core=0.0034785793846155),
 'G.652.D-like (illustrative)':dict(profile='step',core_radius_um=4.1,delta_n_core=0.005),
 'Graded core (illustrative)':dict(profile='graded',core_radius_um=5.0,delta_n_core=0.006,graded_alpha=2.0),
 'Trench-assisted (illustrative)':dict(profile='trench',core_radius_um=4.1,delta_n_core=0.005,trench_start_um=6.0,trench_end_um=9.0,trench_delta_n=-0.001),
 'Multilayer (illustrative)':dict(profile='multilayer',outer_radius_um=40.0,shells=((4.0,0.006),(5.0,0.002),(7.0,0.0),(9.0,-0.0008)))
}


In [2]:
from dataclasses import replace, asdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
m=Model(profile='step',wavelength_nm=1550.,core_radius_um=5.4,
        delta_n_core=0.004,outer_radius_um=40.,n2_m2_per_W=2.2e-20)
n_elements=750
sol=solve_lp01(m,elements=n_elements)
D1550=chromatic_dispersion(m,elements=n_elements,h_nm=10.)
print('MFD_PII_um:',round(sol['MFD_PII_um'],6))
print('MFD_2nd_um:',round(sol['MFD_2nd_um'],6))
print('Aeff_um2:',round(sol['Aeff_um2'],6))
print('D1550_ps_nm_km:',round(D1550,6))
print('neff:',round(sol['neff'],9))
print('gamma_Wkm:',round(sol['gamma_Wkm'],6))

MFD_PII_um: 11.834031
MFD_2nd_um: 12.085655
Aeff_um2: 110.367841
D1550_ps_nm_km: 20.103068
neff: 1.446099217
gamma_Wkm: 0.808032


## 01 / GEOMETRY — 광섬유 단면 및 LP₀₁ 모드 강도

![실행한 모델의 2D 모드 / 굴절률 분포](assets/mode_index.svg)

**왼쪽:** 가운데가 밝을수록 LP₀₁ 광 강도가 큽니다. 원의 점선은 코어 경계(반경 5.4 µm)입니다. **MFD 11.834 µm는 코어 직경 10.8 µm와 동일한 값이 아닙니다.** 빛의 꼬리는 코어를 넘어 클래딩까지 분포합니다.

아래 셀은 **현재 계산값의 1D FEM 필드**를 반경 방향으로 보간하여 2D로 회전시켜 그립니다. **2D FEM을 재실행한 그림은 아닙니다.**

In [ ]:
# LP01 1D FEM eigenmode -> normalized intensity 2D rotation for visualization
from matplotlib.patches import Circle
radial_r = np.asarray(sol['r_um'])
radial_I = np.abs(np.asarray(sol['field']))**2
radial_I = radial_I / radial_I.max()
coord = np.linspace(-20, 20, 340)  # displayed window, NOT FEM boundary R=40 µm
XX, YY = np.meshgrid(coord, coord)
rho = np.sqrt(XX**2 + YY**2)
I2D = np.interp(rho.ravel(), radial_r, radial_I, right=0).reshape(rho.shape)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
im = ax[0].imshow(I2D, origin='lower', extent=[-20,20,-20,20],
                  cmap='turbo', vmin=0, vmax=1, interpolation='bilinear')
ax[0].add_patch(Circle((0,0),m.core_radius_um,fill=False,linestyle='--',
                       linewidth=1.8,edgecolor='white',label='core boundary'))
ax[0].set(xlabel='x (µm)',ylabel='y (µm)',title='LP01 normalized intensity — 2D rotation')
ax[0].set_aspect('equal'); ax[0].legend(loc='upper right',fontsize=8)
fig.colorbar(im, ax=ax[0],label='I / Imax')
ax[1].plot(radial_r, radial_I, color='#267bc0',lw=2,label='FEM radial intensity')
ax[1].axvline(m.core_radius_um,color='#c98a29',ls='--',label='core radius 5.4 µm')
ax[1].set(xlim=(0,20),ylim=(0,1.04),xlabel='r (µm)',ylabel='Normalized intensity',
          title='LP01 intensity vs radius');ax[1].grid(alpha=.25);ax[1].legend()
plt.tight_layout();plt.show()

## 02 / REFRACTIVE INDEX — n(r)

**오른쪽 계단형 선의 의미:** r=0~5.4 µm는 코어, 그 밖은 클래딩입니다. 1550 nm에서 순수 실리카의 굴절률 약 1.444024에, 코어 Δn=0.004를 더하면 약 1.448024가 됩니다.

- **Step-index는 가정한 구조**이며 YOFC의 공개된 실제 프로파일이 아닙니다.
- **R=40 µm는 계산 경계**이고 YOFC 표의 외경 125±1 µm와 다른 개념입니다.

In [ ]:
# Refractive index profile for the same physical inputs
r_plot = np.linspace(0, 20, 401)
n_clad = float(silica_index(m.wavelength_nm/1000))
n_core = n_clad + m.delta_n_core
fig,ax = plt.subplots(figsize=(9,3.8))
ax.plot(r_plot,n_clad+delta_n_r(m,r_plot),color='#2888c9',lw=2.5)
ax.axvline(m.core_radius_um,color='#ce9c51',ls='--',label='core boundary')
ax.axhline(n_clad,color='grey',ls=':',label=f'cladding n={n_clad:.6f}')
ax.text(.6,n_core+.00006,f'core n={n_core:.6f}',fontsize=10)
ax.set(xlim=(0,20),xlabel='Radius r (µm)',ylabel='n(r)',
       title='Step-index assumption @1550 nm — not vendor measured RIP')
ax.grid(alpha=.25);ax.legend();plt.tight_layout();plt.show()
print(f'Core radius = {m.core_radius_um} µm, Δn = {m.delta_n_core}, FEM R = {m.outer_radius_um} µm')

## 03 / MODE METRICS — MFD / Aeff / D 해석

| 값 | 무엇을 뜻하나요? | 계산에서 주의할 점 |
|---|---|---|
| **n_eff** | 고유모드의 전파상수 β를 k0로 나눈 값 | 제조사 사양표에는 n_eff 기준이 없음 |
| **MFD_PII** | 모드의 퍼짐을 나타내는 Petermann-II 직경 | 실물 MFD 계측 정의를 별도 확인해야 함 |
| **Aeff** | 광 세기 분포로 정해지는 비선형 계산용 면적 | 실제 코어 면적 πa²와 다름 |
| **D** | n_eff(λ) 미분으로 얻는 색분산 | 현재는 도핑별 분산 물성 없이 상수 Δn 가정 |

위 값과 YOFC 기준 비교는 **유사 모델의 사양 적합성 점검**일 뿐, 실물 성능 예측 성적표가 아닙니다.

In [3]:
comparison=pd.DataFrame([
 {'parameter':'MFD_PII @1550','YOFC_reference':'11.4–12.2 (typical 11.8) µm','simulation':sol['MFD_PII_um'],'typical_relative_error_pct':(sol['MFD_PII_um']/11.8-1)*100},
 {'parameter':'Aeff @1550','YOFC_reference':'typical 110 µm² (not acceptance tolerance)','simulation':sol['Aeff_um2'],'typical_relative_error_pct':(sol['Aeff_um2']/110-1)*100},
 {'parameter':'D @1550','YOFC_reference':'≤22 ps/(nm·km)','simulation':D1550,'typical_relative_error_pct':float('nan')}
])
print(comparison.to_string(index=False,float_format=lambda x:f'{x:.5f}'))
print('Caution: target-fitted agreement; not YOFC-specific physical validation')

             parameter                              YOFC_reference  simulation  typical_relative_error_pct
        MFD_PII @1550                11.4–12.2 (typical 11.8) µm    11.83403                     0.28840
           Aeff @1550 typical 110 µm² (not acceptance tolerance)   110.36784                     0.33440
              D @1550                              ≤22 ps/(nm·km)    20.10307                         nan
Caution: target-fitted agreement; not YOFC-specific physical validation


## 04 / MESH CONVERGENCE — 계산값이 안정적인가?

Radial elements를 300→500→750→1000으로 변경하면서 계산합니다. 아래 그림은 이산화 메시 변화에 따른 **수치해의 안정성**을 보여주며, 실제 제품 데이터와 일치하는지 확인하는 시험은 아닙니다.

In [4]:
mesh_rows=[]
for n in (300,500,750,1000):
    q=solve_lp01(m,elements=n)
    mesh_rows.append(dict(elements=n,MFD_PII_um=q['MFD_PII_um'],MFD_2nd_um=q['MFD_2nd_um'],
                     Aeff_um2=q['Aeff_um2'],D1550_ps_nm_km=chromatic_dispersion(m,elements=n,h_nm=10),
                     eig_residual=q['residual']))
mesh_df=pd.DataFrame(mesh_rows)
print(mesh_df[['elements','MFD_PII_um','Aeff_um2','D1550_ps_nm_km']].to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('Aeff 750->1000 relative difference (%):',abs(mesh_df.iloc[2].Aeff_um2/mesh_df.iloc[3].Aeff_um2-1)*100)

 elements  MFD_PII_um   Aeff_um2  D1550_ps_nm_km
      300   11.834270  110.373769        20.102564
      500   11.834089  110.369281        20.102950
      750   11.834031  110.367841        20.103068
     1000   11.834011  110.367343        20.103109
Aeff 750->1000 relative difference (%): 0.0004509


In [ ]:
# Aeff numerical convergence across 300/500/750/1000 radial elements
fig,axs = plt.subplots(1,2,figsize=(11,3.9))
axs[0].plot(mesh_df.elements,mesh_df.Aeff_um2,'o-',color='#3086c6')
axs[0].set(xlabel='Radial elements',ylabel='Aeff (µm²)',title='Aeff vs mesh size')
axs[0].grid(alpha=.25)
axs[1].plot(mesh_df.elements,mesh_df.MFD_PII_um,'o-',color='#2c9c8e')
axs[1].set(xlabel='Radial elements',ylabel='MFD PII (µm)',title='MFD vs mesh size')
axs[1].grid(alpha=.25)
plt.tight_layout();plt.show()
print('Aeff difference 750->1000 (%):',abs(mesh_df.iloc[2].Aeff_um2/mesh_df.iloc[3].Aeff_um2-1)*100)

## 05 / SPECTRAL CHARACTERISTICS — 1550 nm에서 결과·기준 분리

![1550 nm 강조: ● FEM 계산값 / × YOFC 기준값](assets/spectrum_mesh.svg)

각 그래프의 세로 점선과 청록색 X축 눈금은 **1550 nm**입니다.

| 1550 nm 값 | ● FEM 계산 | × YOFC 사양/대표 |
|---|---:|---:|
| MFD_PII | **11.8340 µm** | **11.8 µm** (범위 11.4–12.2) |
| Aeff | **110.3678 µm²** | **110 µm² (Typical)** |
| D | **20.1031 ps/(nm·km)** | **≤22 ps/(nm·km) (@1550만)** |

**중요:** ●는 계산 결과이고, ×는 제조사 기준값이며 별도 실측 점이 아닙니다. MFD/Aeff는 대표값에 맞추어 입력을 설정했으므로 독립 예측 성적이 아닙니다. D 상한 22는 **1550 nm**에 한정되며 1590–1620 nm에 그대로 적용하면 안 됩니다. 모델의 1550 nm 부근 분산 기울기는 약 **0.059088 ps/(nm²·km)**입니다.

**최신 웹 솔버 추가 해석:** S₁₅₅₀와 S₀를 구분하고 λ₀를 실제로 탐색한 결과는 아래 07번 섹션 및 `dispersion_slopes.svg`에서 확인할 수 있습니다. 이전 `S≈0.059088` 값은 **S₁₅₅₀**이지 S₀가 아닙니다.

In [5]:
spectrum_rows=[]
for wave in range(1530,1621,10):
    mm=replace(m,wavelength_nm=float(wave))
    a=solve_lp01(mm,elements=n_elements)
    spectrum_rows.append(dict(wavelength_nm=wave,MFD_PII_um=a['MFD_PII_um'],Aeff_um2=a['Aeff_um2'],
                              D_ps_nm_km=chromatic_dispersion(mm,elements=n_elements,h_nm=5),neff=a['neff']))
spec_df=pd.DataFrame(spectrum_rows)
slope=(spec_df.loc[spec_df.wavelength_nm==1560,'D_ps_nm_km'].iloc[0]-spec_df.loc[spec_df.wavelength_nm==1540,'D_ps_nm_km'].iloc[0])/20
print(spec_df[['wavelength_nm','MFD_PII_um','Aeff_um2','D_ps_nm_km']].to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('D slope near 1550 (ps/nm²/km):',round(slope,7),'(YOFC limit ≤0.070)')

 wavelength_nm  MFD_PII_um   Aeff_um2  D_ps_nm_km
          1530   11.729991  108.713353   18.907629
          1540   11.781837  109.535729   19.508841
          1550   11.834031  110.367841   20.103067
          1560   11.886578  111.209840   20.690610
          1570   11.939480  112.061883   21.271761
          1580   11.992743  112.924126   21.846802
          1590   12.046368  113.796731   22.416007
          1600   12.100361  114.679861   22.979637
          1610   12.154726  115.573681   23.537947
          1620   12.209466  116.478363   24.091183
D slope near 1550 (ps/nm²/km): 0.0590884 (YOFC limit ≤0.070)


In [ ]:
# Explicit 1550 nm computed (●) versus YOFC reference (×) in every panel.
# The YOFC D<=22 specification is applicable at 1550 nm, NOT the full wavelength sweep.
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=(16.8, 5.4))
specs = [
    ('MFD_PII_um', 'MFD_PII (µm)', 11.8, 'YOFC typical 11.8 µm', (11.6, 12.3), '#24b7c4', '11.4–12.2 µm range @1550'),
    ('Aeff_um2', 'Aeff (µm²)', 110., 'YOFC typical 110 µm²', (107, 118), '#5498ed', 'Typical, not pass/fail tolerance'),
    ('D_ps_nm_km', 'D (ps/(nm·km))', 22., 'YOFC upper limit ≤22', (17.5, 25), '#e8a749', 'Upper limit applies @1550 ONLY')
]
for ax, (column, title, reference, ref_name, ylim, color, note) in zip(axs, specs):
    xs = spec_df['wavelength_nm'].to_numpy(dtype=float)
    ys = spec_df[column].to_numpy(dtype=float)
    calc_1550 = float(spec_df.loc[spec_df['wavelength_nm'] == 1550, column].iloc[0])
    ax.axvspan(1548.5, 1551.5, color='#64dce5', alpha=0.1)
    ax.axvline(1550, color='#1399a6', lw=1.5, ls='--', alpha=.85, zorder=1)
    ax.plot(xs, ys, 'o-', lw=2.3, ms=3.6, color=color, label='FEM model (all wavelengths)')
    ax.scatter([1550], [reference], marker='x', s=135, color='#e44367', lw=2.4, zorder=5, label='YOFC reference @1550')
    ax.scatter([1550], [calc_1550], s=95, facecolor='#ffcc7d', edgecolor='black', linewidth=1.3, zorder=6, label='FEM result @1550')
    ax.set_xlim(1528, 1622)
    ax.set_ylim(*ylim)
    ax.set_xticks([1530, 1540, 1550, 1560, 1590, 1620])
    ax.set(xlabel='Wavelength (nm)', ylabel=title, title=title + ' vs wavelength')
    ax.grid(alpha=.22)
    ax.text(.04, .98,
            f'● FEM @1550 = {calc_1550:.4f}\n× {ref_name} @1550 = {reference:g}\n{note}',
            transform=ax.transAxes, va='top', ha='left', fontsize=8.5,
            bbox=dict(facecolor='white', edgecolor='#b8cbd9', alpha=.92, boxstyle='round,pad=.48'))
    for tick in ax.get_xticklabels():
        if tick.get_text() == '1550':
            tick.set_color('#0a8e9b')
            tick.set_fontweight('bold')
    ax.legend(fontsize=7.2, loc='lower right')
fig.suptitle('1550 nm comparison: ● calculated FEM / × YOFC reference (not a measured point)', fontsize=13, weight='bold')
plt.tight_layout(rect=(0, 0, 1, .95))
plt.show()
print('1550 nm values: MFD=', float(spec_df.loc[spec_df.wavelength_nm == 1550, 'MFD_PII_um'].iloc[0]),
      'Aeff=', float(spec_df.loc[spec_df.wavelength_nm == 1550, 'Aeff_um2'].iloc[0]),
      'D=', float(spec_df.loc[spec_df.wavelength_nm == 1550, 'D_ps_nm_km'].iloc[0]),
      'YOFC D limit @1550 = 22')

In [6]:
outdir=Path('validation_outputs');outdir.mkdir(exist_ok=True)
comparison.to_csv(outdir/'comparison.csv',index=False)
mesh_df.to_csv(outdir/'mesh_convergence.csv',index=False)
spec_df.to_csv(outdir/'spectrum.csv',index=False)
summary={'method':'scalar LP01 radial P1 FEM, fitted illustrative step-index',
         'core_radius_um':m.core_radius_um,'core_delta_n':m.delta_n_core,
         'elements':n_elements,'MFD_PII_um':float(sol['MFD_PII_um']),
         'MFD_2nd_um':float(sol['MFD_2nd_um']),'Aeff_um2':float(sol['Aeff_um2']),
         'D1550_ps_nm_km':float(D1550),'slope_ps_nm2_km':float(slope)}
(outdir/'summary.json').write_text(json.dumps(summary,indent=2))
print('Generated comparison.csv, mesh_convergence.csv, spectrum.csv, summary.json')

Generated comparison.csv, mesh_convergence.csv, spectrum.csv, summary.json


## 06 / LIMITATIONS — 실제 YOFC 광섬유에 대해 말할 수 없는 것

- **이 노트북이 계산한 것:** Step-index 유사 모델의 LP₀₁ eigenmode, n_eff, MFD, Aeff, D, dD/dλ, 메시 수렴.
- **사양에 맞추기 위해 입력을 조정한 것:** 코어 반경 5.4 µm, Δn=0.004 → MFD≈11.8, Aeff≈110.
- **계산하지 않은 것:** 제품의 감쇠, 케이블 컷오프, PMD, 굽힘 손실, 외경 공차, 도핑별 정확한 분산, 온도/제조 편차.
- **실물 검증에 필요한 것:** 측정된 반경별 굴절률 프로파일(RIP)과 복수 파장 MFD/D 실측값.
- **COMSOL Wave Optics 실행 아님:** 동일 계열의 원통 대칭 Python/SciPy P1 FEM 해석.

[한눈에 보는 그래픽 보고서](./VALIDATION_REPORT.md)

**2026-11-03 v0.2.1 업데이트:** 기존 단일 `dispersion_slope`를 S₁₅₅₀(1550 nm)와 S₀(영분산 파장 λ₀)로 구분. YOFC는 본 자료에서 λ₀와 S₀ 직접 수치를 제공하지 않았습니다.

## 07 / 최신 웹 솔버 v0.2.1 — 직접 재실행 데이터

GitHub Pages의 현재 **동일한 `solver.mjs` 코드**를 실행한 결과를 재사용 가능한 JSON과 함께 보관합니다. 기존 Python SciPy 실행 결과(앞 부분)와 web 0.2.1 출력값이 일치하는지 확인하십시오.

| 구분 | 모델 계산 | 제조사 사양표 |
|---|---:|---|
| **S₁₅₅₀** (1550 nm 분산 기울기) | **0.059089** ps/(nm²·km) | **≤0.070 @1550 nm** |
| **λ₀** (D=0인 영분산 파장) | **1275.457 nm** | 제시되지 않음 |
| **S₀** (λ₀에서의 기울기) | **0.093979** ps/(nm²·km) | 제시되지 않음 |

![S0 vs S1550](assets/dispersion_slopes.svg)

*중요: YOFC의 `≤0.070`을 S₀에 적용하지 마십시오. 프로그램의 분산 기울기는 `S1550=(D(1560)-D(1540))/20`, `S0=(D(λ0+10)-D(λ0-10))/20`이며, D 계산에는 n_eff의 5점 파장 차분(5 nm)이 쓰입니다.*

In [ ]:
# 2026-11-03: already executed result of the exact current web engine (v0.2.1)
# This embedded record was calculated using the GitHub solver.mjs and saved to web_solver_rerun.json.
import json, pandas as pd, matplotlib.pyplot as plt
web_snapshot = json.loads(r'''{"timestamp_date":"2026-11-03","engine":"Universal Silica Fiber Web v0.2.1","solver_repository_file":"COMSOL/universal_silica_v01/web/solver.mjs","solver_blob_sha":"a05fc21585d1d164bb95ef0db7abd97547acae9f","method":"axisymmetric scalar LP01 radial P1 FEM, silica Sellmeier plus wavelength-independent delta n","actual_product_validation":false,"step_index_fitted_to_MFD_Aeff":true,"model":{"profile":"step","wavelength_nm":1550,"core_radius_um":5.4,"delta_n_core":0.004,"outer_radius_um":40,"graded_alpha":2,"trench_start_um":7,"trench_end_um":10,"trench_delta_n":-0.001,"shells":[[4,0.006],[5,0.002],[7,0],[9,-0.0008]],"n2_m2_per_W":2.2e-20,"elements":750},"output":{"n_eff":1.446099217459323,"MFD_PII_um":11.834031125189526,"MFD_2nd_um":12.085654964898618,"Aeff_um2":110.36784066727965,"gamma_Wkm":0.8080315256995695,"D1550_ps_nm_km":20.103067631843608,"eigen_residual":1.0016128413366348e-12,"guided_proxy":true,"actual_mesh_elements":750,"slope_1550_ps_nm2_km":0.0590885086559906,"lambda0_nm":1275.457178224494,"slope_zero_ps_nm2_km":0.0939788195507043,"zero_found":true},"dispersion_slope_method":{"scan_range_nm":[1100,1550],"neff_finite_difference_step_nm":5,"slope_center_step_nm":10,"s1550_formula":"(D(1560 nm)-D(1540 nm))/20 nm","S0_formula":"(D(lambda0+10 nm)-D(lambda0-10 nm))/20 nm; lambda0 root of D(lambda)"},"manufacturer_reference":{"basis":"provided screenshot","MFD_min_um":11.4,"MFD_max_um":12.2,"MFD_typical_um":11.8,"Aeff_typical_um2":110,"D1550_max_ps_nm_km":22,"S1550_max_ps_nm2_km":0.07,"S0_reference":null,"lambda0_reference":null},"fitted_model_mesh_relative_Aeff_pct_750_vs_1000":0.0004509317390276513,"spectrum":[{"wavelength_nm":1530,"n_eff":1.446369319729971,"MFD_PII_um":11.729990885364831,"Aeff_um2":108.71335309472956,"D_ps_nm_km":18.907628712388867,"gamma_Wkm":0.8310520540988954},{"wavelength_nm":1540,"n_eff":1.44623445847374,"MFD_PII_um":11.781836742073075,"Aeff_um2":109.53572895124694,"D_ps_nm_km":19.50884136289246,"gamma_Wkm":0.8194567285211252},{"wavelength_nm":1550,"n_eff":1.446099217459323,"MFD_PII_um":11.834031125189526,"Aeff_um2":110.36784066727965,"D_ps_nm_km":20.103067631843608,"gamma_Wkm":0.8080315256995695},{"wavelength_nm":1560,"n_eff":1.4459635876428187,"MFD_PII_um":11.886577719727605,"Aeff_um2":111.20984040974179,"D_ps_nm_km":20.690610114288646,"gamma_Wkm":0.7967732283071152},{"wavelength_nm":1570,"n_eff":1.4458275602247477,"MFD_PII_um":11.939480254617138,"Aeff_um2":112.06188310589212,"D_ps_nm_km":21.2717608928367,"gamma_Wkm":0.7856787021370911},{"wavelength_nm":1580,"n_eff":1.4456911266398018,"MFD_PII_um":11.992742502027609,"Aeff_um2":112.92412648182203,"D_ps_nm_km":21.846802438315883,"gamma_Wkm":0.7747448935743309},{"wavelength_nm":1590,"n_eff":1.4455542785470383,"MFD_PII_um":12.046368279479843,"Aeff_um2":113.79673114341325,"D_ps_nm_km":22.416006947523574,"gamma_Wkm":0.763968826873564},{"wavelength_nm":1600,"n_eff":1.4454170078205124,"MFD_PII_um":12.100361448992969,"Aeff_um2":114.67986061433112,"D_ps_nm_km":22.979637456439967,"gamma_Wkm":0.7533476018449484},{"wavelength_nm":1610,"n_eff":1.4452793065403176,"MFD_PII_um":12.154725919247323,"Aeff_um2":115.57368142060258,"D_ps_nm_km":23.537947827285066,"gamma_Wkm":0.742878391321915},{"wavelength_nm":1620,"n_eff":1.4451411669840173,"MFD_PII_um":12.209465644690077,"Aeff_um2":116.47836313011581,"D_ps_nm_km":24.091183314254106,"gamma_Wkm":0.7325584390292373}],"mesh":[{"requested_elements":300,"elements":301,"n_eff":1.4460990718278826,"MFD_PII_um":11.834271126797107,"Aeff_um2":110.37382179505532,"residual":1.00797466566086e-12},{"requested_elements":500,"elements":501,"n_eff":1.4460991827557967,"MFD_PII_um":11.834088345907167,"Aeff_um2":110.36926775635276,"residual":1.0052158136625079e-12},{"requested_elements":750,"elements":750,"n_eff":1.446099217459323,"MFD_PII_um":11.834031125189526,"Aeff_um2":110.36784066727965,"residual":1.0016128413366348e-12},{"requested_elements":1000,"elements":1000,"n_eff":1.4460992297348418,"MFD_PII_um":11.834011114558304,"Aeff_um2":110.3673429859006,"residual":5.726881113683308e-13}],"dispersion_curve":[{"wavelength_nm":1100,"D_ps_nm_km":-20.838029870019305},{"wavelength_nm":1120,"D_ps_nm_km":-17.909515496152263},{"wavelength_nm":1140,"D_ps_nm_km":-15.153524179779032},{"wavelength_nm":1160,"D_ps_nm_km":-12.554672474215181},{"wavelength_nm":1180,"D_ps_nm_km":-10.099139708540427},{"wavelength_nm":1200,"D_ps_nm_km":-7.774475544840457},{"wavelength_nm":1220,"D_ps_nm_km":-5.569451753705077},{"wavelength_nm":1240,"D_ps_nm_km":-3.4739129281121697},{"wavelength_nm":1260,"D_ps_nm_km":-1.4786584583937232},{"wavelength_nm":1275.457178224494,"D_ps_nm_km":-6.140429670331282e-7},{"wavelength_nm":1280,"D_ps_nm_km":0.42465544606539873},{"wavelength_nm":1300,"D_ps_nm_km":2.243621750118486},{"wavelength_nm":1320,"D_ps_nm_km":3.985149931046378},{"wavelength_nm":1340,"D_ps_nm_km":5.655551682623326},{"wavelength_nm":1360,"D_ps_nm_km":7.260588138185967},{"wavelength_nm":1380,"D_ps_nm_km":8.805537202502409},{"wavelength_nm":1400,"D_ps_nm_km":10.29523969619179},{"wavelength_nm":1420,"D_ps_nm_km":11.734140038212775},{"wavelength_nm":1440,"D_ps_nm_km":13.12632743377919},{"wavelength_nm":1460,"D_ps_nm_km":14.475569441959461},{"wavelength_nm":1480,"D_ps_nm_km":15.78534657249868},{"wavelength_nm":1500,"D_ps_nm_km":17.05886912647642},{"wavelength_nm":1520,"D_ps_nm_km":18.299116684898955},{"wavelength_nm":1540,"D_ps_nm_km":19.508839402930427},{"wavelength_nm":1560,"D_ps_nm_km":20.69060957605024},{"wavelength_nm":1580,"D_ps_nm_km":21.84680163864881},{"wavelength_nm":1600,"D_ps_nm_km":22.97963810032133},{"wavelength_nm":1620,"D_ps_nm_km":24.091182193374664}]}''')
wo = web_snapshot['output']
table = pd.DataFrame([
 {'parameter':'MFD_PII @1550 (µm)','web_result':wo['MFD_PII_um'],'YOFC_reference':'11.4–12.2 (typical 11.8)'},
 {'parameter':'Aeff @1550 (µm²)','web_result':wo['Aeff_um2'],'YOFC_reference':'typical 110; no acceptance tolerance'},
 {'parameter':'D @1550 (ps/nm/km)','web_result':wo['D1550_ps_nm_km'],'YOFC_reference':'≤22'},
 {'parameter':'S1550 (ps/nm²/km)','web_result':wo['slope_1550_ps_nm2_km'],'YOFC_reference':'≤0.070 at 1550 nm'},
 {'parameter':'lambda0 (nm)','web_result':wo['lambda0_nm'],'YOFC_reference':'not provided'},
 {'parameter':'S0 (ps/nm²/km)','web_result':wo['slope_zero_ps_nm2_km'],'YOFC_reference':'not provided'}
])
print('Saved v0.2.1 web-engine execution snapshot:')
print(table.to_string(index=False,formatters={'web_result':lambda x:f'{x:.6f}'}))
assert abs(wo['slope_1550_ps_nm2_km']-wo['slope_zero_ps_nm2_km']) > 0.02


## 08 / Current Web solver source embedded — reproducible exact rerun

아래 셀에는 최신 웹 솔버의 **정확한 JS 소스**(GitHub blob `a05fc21585d1d164bb95ef0db7abd97547acae9f`)를 포함했습니다. **Node.js ≥18**이 설치된 환경에서는 동일한 계산을 새로 실행한 뒤 저장된 결과와 비교합니다. Node가 없는 환경에서도 상단 Python/SciPy 계산 및 기록된 실제 웹 솔버 결과를 확인할 수 있습니다. 외부 네트워크 연결은 필요하지 않습니다.

In [ ]:
# Exact local rerun of current web/solver.mjs (source bundled; no network calls).
# Requires Node.js >= 18 on this runtime. Python SciPy cells above work independently.
import shutil, subprocess, tempfile
from pathlib import Path
WEB_SOLVER_SOURCE = r'''/**
 * Universal Silica Fiber v0.2 — client-side radial P1 FEM.
 * Mirrors the physical model in universal_silica_v01/universal_radial.py.
 * Only rotationally symmetric, weak-guidance scalar LP0 (dominant LP01).
 * Units: transverse micrometres, wavelength nanometres, n2 m²/W.
 * No COMSOL code, server, static lookup, or manufacturer-profile claim.
 */
export const VERSION = '0.2.1';
export const C0 = 299792458;
const B=[0.6961663,0.4079426,0.8974794];
const C=[0.0684043**2,0.1162414**2,9.896161**2];
const qx=[0.046910077030668,0.230765344947158,0.5,0.769234655052841,0.953089922969332];
const qw=[0.118463442528095,0.239314335249683,0.284444444444444,0.239314335249683,0.118463442528095];
export const PRESETS = {
  'g654':{profile:'step', wavelength_nm:1550,core_radius_um:5.5,delta_n_core:0.0034785793846155,outer_radius_um:40,graded_alpha:2,trench_start_um:7,trench_end_um:10,trench_delta_n:-0.001,shells:[[4,0.006],[5,0.002],[7,0],[9,-0.0008]],n2_m2_per_W:2.2e-20,elements:750},
  'g652':{profile:'step',wavelength_nm:1550,core_radius_um:4.1,delta_n_core:0.005,outer_radius_um:40,graded_alpha:2,trench_start_um:7,trench_end_um:10,trench_delta_n:-0.001,shells:[[4,0.006],[5,0.002],[7,0],[9,-0.0008]],n2_m2_per_W:2.6e-20,elements:750},
  'graded':{profile:'graded',wavelength_nm:1550,core_radius_um:5.0,delta_n_core:0.006,outer_radius_um:40,graded_alpha:2,trench_start_um:7,trench_end_um:10,trench_delta_n:-0.001,shells:[[4,0.006],[5,0.002],[7,0],[9,-0.0008]],n2_m2_per_W:2.6e-20,elements:750},
  'trench':{profile:'trench',wavelength_nm:1550,core_radius_um:4.1,delta_n_core:0.005,outer_radius_um:40,graded_alpha:2,trench_start_um:6,trench_end_um:9,trench_delta_n:-0.001,shells:[[4,0.006],[5,0.002],[7,0],[9,-0.0008]],n2_m2_per_W:2.6e-20,elements:750},
  'multi':{profile:'multilayer',wavelength_nm:1550,core_radius_um:4.0,delta_n_core:0.006,outer_radius_um:40,graded_alpha:2,trench_start_um:7,trench_end_um:10,trench_delta_n:-0.001,shells:[[4,0.006],[5,0.002],[7,0],[9,-0.0008]],n2_m2_per_W:2.6e-20,elements:750}
};
export function silicaIndex(wavelengthNm){const w=Number(wavelengthNm)/1000;if(!Number.isFinite(w)||w<=0.21||w>=3.7)throw Error('Silica Sellmeier 유효 범위는 210~3700 nm입니다.'); const w2=w*w;return Math.sqrt(1+B.reduce((v,b,i)=>v+b*w2/(w2-C[i]),0));}
export function validate(m){
 if(!['step','graded','trench','multilayer'].includes(m.profile))throw Error('지원하지 않는 단면입니다.');
 for(const [key,min] of [['wavelength_nm',210],['core_radius_um',0],['outer_radius_um',0],['graded_alpha',0],['n2_m2_per_W',0]]){if(!(Number.isFinite(m[key])&&m[key]>min))throw Error(`${key} 입력값을 확인하세요.`);}
 if(m.wavelength_nm>=3700)throw Error('파장이 3700 nm를 초과합니다.');
 if(m.outer_radius_um<=2.2*m.core_radius_um)throw Error('계산 영역 반경은 코어 반경의 2.2배보다 커야 합니다.');
 if(!Number.isFinite(m.delta_n_core)||!Number.isFinite(m.trench_delta_n))throw Error('굴절률 차이가 유효하지 않습니다.');
 if(!Number.isInteger(m.elements)||m.elements<80||m.elements>5000)throw Error('Mesh 수는 80~5000 사이의 정수여야 합니다.');
 if(m.profile==='trench'&&!(m.core_radius_um<m.trench_start_um&&m.trench_start_um<m.trench_end_um&&m.trench_end_um<m.outer_radius_um))throw Error('Trench 경계는 코어 < 시작 < 종료 < 계산 영역이어야 합니다.');
 if(m.profile==='multilayer'){
   if(!Array.isArray(m.shells)||!m.shells.length)throw Error('다층 쉘 데이터를 입력하세요.');
   let p=0;for(const [end,dn] of m.shells){if(!(Number.isFinite(end)&&end>p&&end<m.outer_radius_um&&Number.isFinite(dn)))throw Error('쉘 외경은 증가해야 하며 계산 영역보다 작아야 합니다.');p=end;}
 }
 const ncl=silicaIndex(m.wavelength_nm);let maxDn=m.delta_n_core;
 if(m.profile==='multilayer')maxDn=Math.max(...m.shells.map(x=>x[1]));
 if(maxDn<=0)throw Error('기본 유도 모드를 위한 양의 코어 Δn이 필요합니다.');
 if(m.profile==='trench'&&m.trench_delta_n>=0)throw Error('Trench Δn은 음수로 입력하세요.');
 if(ncl+Math.min(-0.01,m.trench_delta_n)<0)throw Error('물리적으로 유효하지 않은 굴절률입니다.');
 return m;
}
export function edges(m){const arr=m.profile==='trench'?[m.core_radius_um,m.trench_start_um,m.trench_end_um]:m.profile==='multilayer'?m.shells.map(x=>x[0]):[m.core_radius_um];return [0,...arr,m.outer_radius_um];}
export function deltaIndex(m,r){switch(m.profile){case 'step':return r<m.core_radius_um?m.delta_n_core:0;case 'graded':return r<m.core_radius_um?m.delta_n_core*Math.max(0,1-(r/m.core_radius_um)**m.graded_alpha):0;case 'trench':return r<m.core_radius_um?m.delta_n_core:(r>=m.trench_start_um&&r<m.trench_end_um?m.trench_delta_n:0);case 'multilayer':{let p=0;for(const [end,dn] of m.shells){if(r>=p&&r<end)return dn;p=end;}return 0;}default:return 0;}}
export function radialGrid(m){const br=edges(m), intervals=br.slice(1).map((x,i)=>x-br[i]);const total=intervals.reduce((a,b)=>a+b,0);const r=[];for(let j=0;j<intervals.length;j++){const n=Math.max(12,Math.round(m.elements*intervals[j]/total));for(let i=0;i<n;i++)r.push(br[j]+intervals[j]*i/n);}r.push(br.at(-1));return Float64Array.from(r);}
function tridiagSolve(diag,off,rhs){
 const N=diag.length;const p=new Float64Array(N),z=new Float64Array(N);
 let d=diag[0];if(!(d>0))throw Error('FEM 선형 시스템이 양의 정부호가 아닙니다.');p[0]=N>1?off[0]/d:0;z[0]=rhs[0]/d;
 for(let i=1;i<N;i++){d=diag[i]-off[i-1]*p[i-1];if(!(d>0))throw Error('Cholesky 분해 실패: σ 또는 구조 조건을 확인하세요.');p[i]=i<N-1?off[i]/d:0;z[i]=(rhs[i]-off[i-1]*z[i-1])/d;}
 for(let i=N-2;i>=0;i--)z[i]-=p[i]*z[i+1];return z;
}
function tridiagMv(d,o,x){const n=d.length,y=new Float64Array(n);for(let i=0;i<n;i++)y[i]=d[i]*x[i]+(i?o[i-1]*x[i-1]:0)+(i<n-1?o[i]*x[i+1]:0);return y;}
function dot(a,b){let s=0;for(let i=0;i<a.length;i++)s+=a[i]*b[i];return s;}
export function solve(m){
 validate(m);
 const r=radialGrid(m),N=r.length-1,k0=2*Math.PI/(m.wavelength_nm/1000),ncl=silicaIndex(m.wavelength_nm);
 const ad=new Float64Array(N),ao=new Float64Array(N-1),bd=new Float64Array(N),bo=new Float64Array(N-1);
 let maxDn=-Infinity;
 for(let e=0;e<N;e++){
   const r0=r[e],h=r[e+1]-r0;
   const stiff=(r[e+1]*r[e+1]-r0*r0)/(2*h*h);
   let mass00=0,mass01=0,mass11=0,n00=0,n01=0,n11=0;
   for(let j=0;j<5;j++){
      const t=qx[j],radius=r0+h*t,wt=qw[j]*h*radius,pa=1-t,pb=t;const dn=deltaIndex(m,radius);const n=ncl+dn;
      maxDn=Math.max(maxDn,dn);
      mass00+=wt*pa*pa;mass01+=wt*pa*pb;mass11+=wt*pb*pb;
      n00+=wt*n*n*pa*pa;n01+=wt*n*n*pa*pb;n11+=wt*n*n*pb*pb;
   }
   ad[e]+=k0*k0*n00-stiff;bd[e]+=mass00;
   if(e+1<N){ad[e+1]+=k0*k0*n11-stiff;bd[e+1]+=mass11;ao[e]+=k0*k0*n01+stiff;bo[e]+=mass01;}
 }
 const sigma=(k0*(ncl+maxDn))**2+0.05;
 const hd=Float64Array.from(ad,(x,i)=>sigma*bd[i]-x),ho=Float64Array.from(ao,(x,i)=>sigma*bo[i]-x);
 let x=new Float64Array(N).fill(1),lambda=0,iter=0,conv=false;
 const norm=x=>Math.sqrt(dot(x,tridiagMv(bd,bo,x)));
 {const s=norm(x);for(let i=0;i<N;i++)x[i]/=s;}
 for(iter=0;iter<180;iter++){
   const rhs=tridiagMv(bd,bo,x),y=tridiagSolve(hd,ho,rhs),sn=norm(y);
   for(let i=0;i<N;i++)y[i]/=sn;
   if(y[0]<0)for(let i=0;i<N;i++)y[i]=-y[i];
   let dx=0;for(let i=0;i<N;i++){const d=y[i]-x[i];dx+=d*d;}
   x=y;
   if(dx<1e-20){conv=true;break;}
 }
 const Ax=tridiagMv(ad,ao,x),Bx=tridiagMv(bd,bo,x);
 lambda=dot(x,Ax)/dot(x,Bx);if(!(lambda>0))throw Error('양의 β² 고유모드를 찾지 못했습니다.');
 let residual2=0,den2=0;for(let i=0;i<N;i++){residual2+=(Ax[i]-lambda*Bx[i])**2;den2+=(lambda*Bx[i])**2;}
 const residual=Math.sqrt(residual2/den2),neff=Math.sqrt(lambda)/k0;
 const psi=new Float64Array(r.length);psi.set(x);
 let P=0,Q=0,Rs=0,grad=0,P0=0;
 for(let e=0;e<N;e++){
   const h=r[e+1]-r[e],slope=(psi[e+1]-psi[e])/h;
   for(let j=0;j<5;j++){
      const t=qx[j],rad=r[e]+h*t,wt=h*qw[j]*rad,field=(1-t)*psi[e]+t*psi[e+1],intensity=field*field;
      P0+=wt*intensity;P+=2*Math.PI*wt*intensity;Q+=2*Math.PI*wt*intensity*intensity;
      Rs+=2*Math.PI*wt*intensity*rad*rad;grad+=wt*slope*slope;
   }
 }
 if(!(P>0&&Q>0&&grad>0))throw Error('모드 적분에 실패했습니다.');
 const Aeff=P*P/Q, MFD2=2*Math.sqrt(2*Rs/P),MFDPII=2*Math.sqrt(2*P0/grad);
 const normP=Math.sqrt(P);for(let i=0;i<psi.length;i++)psi[i]/=normP;
 const gamma=2*Math.PI/(m.wavelength_nm*1e-9)*m.n2_m2_per_W/(Aeff*1e-12)*1000;
 const guided=neff>ncl&&neff<ncl+maxDn+1e-7;
 return {profile:m.profile,wavelength_nm:m.wavelength_nm,n_clad:ncl,n_eff:neff,n_max:ncl+maxDn,Aeff_um2:Aeff,MFD_2nd_um:MFD2,MFD_PII_um:MFDPII,gamma_Wkm:gamma,residual,elements:N,iterations:iter+1,iteration_converged:conv,guided_proxy:guided,r:Array.from(r),field:Array.from(psi)};
}
export function dispersion(m,hNm=10){
 if(!(hNm>0&&m.wavelength_nm-2*hNm>211&&m.wavelength_nm+2*hNm<3690))throw Error('파장 미분 간격을 줄이세요.');
 const ns=[-2,-1,0,1,2].map(j=>solve({...m,wavelength_nm:m.wavelength_nm+j*hNm}).n_eff);
 const d2=(-ns[4]+16*ns[3]-30*ns[2]+16*ns[1]-ns[0])/(12*(hNm/1000)**2);
 return -(m.wavelength_nm/1000)/C0*1e12*d2;
}
export function spectrum(m,start,end,step){
 if(!(Number.isFinite(start)&&Number.isFinite(end)&&Number.isFinite(step)&&start<end&&step>=2&&end-start<=600))throw Error('스윕 범위(최대 600 nm) 및 스텝(최소 2 nm)을 확인하세요.');
 const count=Math.floor((end-start)/step)+1;if(count>70)throw Error('파장 스윕은 최대 70포인트입니다.');
 const rows=[];for(let i=0;i<count;i++){
  const nm=start+i*step;if(nm-2*step<=211||nm+2*step>=3690)throw Error('스윕 범위가 재료 모델 파장 제한 밖입니다.');
  const mm={...m,wavelength_nm:nm}; const s=solve(mm);
  const D=dispersion(mm,step);
  rows.push({wavelength_nm:nm,n_eff:s.n_eff,MFD_PII_um:s.MFD_PII_um,Aeff_um2:s.Aeff_um2,D_ps_nm_km:D,gamma_Wkm:s.gamma_Wkm});
 }
 return rows;
}
export function meshStudy(m,levels=[200,350,550,850]){
 return levels.map(el=>{const x=solve({...m,elements:el});return {elements:x.elements,n_eff:x.n_eff,MFD_PII_um:x.MFD_PII_um,Aeff_um2:x.Aeff_um2,residual:x.residual};});
}
/**
 * Dispersion slopes for the current hypothetical circular silica-index model.
 * S_1550 = dD/dlambda at 1550 nm, S0 = dD/dlambda where D(lambda0)=0.
 * The two quantities must never be used interchangeably.
 * Not a measured manufacturer RIP or product qualification.
 * Differentiation uses the same five-point n_eff stencil as dispersion().
 */
export function dispersionDiagnostics(m, opts={}) {
 validate(m);
 const from=opts.fromNm??1100, to=opts.toNm??1550;
 const step=opts.scanStepNm??50, h=opts.diffStepNm??5, ds=opts.slopeStepNm??10;
 if(!(from>=250 && to<=3400 && to>from && step>=5 && step<=150 && h>=2 && ds>=5))
  throw Error('영분산 스캔 범위/미분 간격을 확인하세요.');
 const neffCache=new Map(), dCache=new Map();
 const nAt=nm=>{
  const k=nm.toFixed(8);
  if(!neffCache.has(k)){
   const s=solve({...m,wavelength_nm:nm});
   if(!s.guided_proxy || !Number.isFinite(s.n_eff))throw Error('파장 스윕에서 유도 모드를 찾지 못했습니다.');
   neffCache.set(k,s.n_eff);
  }
  return neffCache.get(k);
 };
 const dAt=nm=>{
  const k=nm.toFixed(8);
  if(!dCache.has(k)){
   if(nm-2*h<=211||nm+2*h>=3690)throw Error('분산 계산이 Sellmeier 파장 유효범위를 벗어납니다.');
   const vals=[-2,-1,0,1,2].map(j=>nAt(nm+j*h));
   const d2=(-vals[4]+16*vals[3]-30*vals[2]+16*vals[1]-vals[0])/(12*(h/1000)**2);
   dCache.set(k,-(nm/1000)/C0*1e12*d2);
  }
  return dCache.get(k);
 };
 const slopeAt=nm=>(dAt(nm+ds)-dAt(nm-ds))/(2*ds);
 const s1550=slopeAt(1550);
 let low=from, dl=dAt(low), hi=null, dh=null;
 for(let nm=from+step; nm<=to+1e-9; nm+=step){
   const dv=dAt(nm);
   if(dl===0 || dv===0 || dl*dv<0){hi=nm;dh=dv;break;}
   low=nm;dl=dv;
 }
 if(hi===null) {
  return {slope_1550_ps_nm2_km:s1550,lambda0_nm:null,slope_zero_ps_nm2_km:null,
   zero_found:false,search_nm:[from,to],difference_step_nm:ds,neff_step_nm:h};
 }
 let root=null;
 if(dl===0)root=low;
 else if(dh===0)root=hi;
 else for(let i=0;i<16;i++){
   const sec=(low*dh-hi*dl)/(dh-dl);
   const mid=(low+hi)/2;
   const x=Number.isFinite(sec)&&sec>low+0.05*(hi-low)&&sec<hi-0.05*(hi-low)?sec:mid;
   const d=dAt(x);
   if(Math.abs(d)<1e-4 || hi-low<0.025){root=x;break;}
   if(dl*d<=0){hi=x;dh=d;}else{low=x;dl=d;}
 }
 if(root===null)root=(low+hi)/2;
 return {slope_1550_ps_nm2_km:s1550,lambda0_nm:root,slope_zero_ps_nm2_km:slopeAt(root),
  zero_found:true,search_nm:[from,to],difference_step_nm:ds,neff_step_nm:h};
}

export function outputSummary(sol){const {r,field,...rest}=sol;return rest;}
'''
runner = r'''import {PRESETS,solve,dispersion,dispersionDiagnostics,spectrum,meshStudy} from './solver.mjs';
const m={...structuredClone(PRESETS.g654),core_radius_um:5.4,delta_n_core:0.004};
const s=solve(m);const d=dispersion(m);const diag=dispersionDiagnostics(m);
const spectrumRows=spectrum(m,1530,1620,10);
const meshRows=[300,500,750,1000].map(n=>({requested_elements:n,...meshStudy(m,[n])[0]}));
const ds=[];for(let nm=1100;nm<=1620;nm+=20)ds.push({wavelength_nm:nm,D_ps_nm_km:dispersion({...m,wavelength_nm:nm},5)});
ds.push({wavelength_nm:diag.lambda0_nm,D_ps_nm_km:dispersion({...m,wavelength_nm:diag.lambda0_nm},5)});
ds.sort((a,b)=>a.wavelength_nm-b.wavelength_nm);
const output={
  n_eff:s.n_eff,MFD_PII_um:s.MFD_PII_um,MFD_2nd_um:s.MFD_2nd_um,
  Aeff_um2:s.Aeff_um2,gamma_Wkm:s.gamma_Wkm,D1550_ps_nm_km:d,
  eigen_residual:s.residual,guided_proxy:s.guided_proxy,actual_mesh_elements:s.elements,
  slope_1550_ps_nm2_km:diag.slope_1550_ps_nm2_km,
  lambda0_nm:diag.lambda0_nm,slope_zero_ps_nm2_km:diag.slope_zero_ps_nm2_km,
  zero_found:diag.zero_found
};
process.stdout.write(JSON.stringify({model:m,output,spectrum:spectrumRows,mesh:meshRows,dispersion_curve:ds}));
'''
if shutil.which('node') is None:
    print("Node.js >=18 is not available. The archived *actual web-engine execution* above remains inspectable.")
    print("To run this exact JavaScript engine in Colab, install Node.js and rerun this cell.")
    web_recomputed = None
else:
    with tempfile.TemporaryDirectory() as temp_dir:
        Path(temp_dir,'solver.mjs').write_text(WEB_SOLVER_SOURCE,encoding='utf-8')
        cmd=subprocess.run(['node','--input-type=module','-e',runner],cwd=temp_dir,
                           check=True,text=True,capture_output=True,timeout=120)
        web_recomputed=json.loads(cmd.stdout)
    print('Exact current web solver execution completed using Node.js:')
    for key in ['n_eff','MFD_PII_um','Aeff_um2','D1550_ps_nm_km','slope_1550_ps_nm2_km','lambda0_nm','slope_zero_ps_nm2_km']:
        recorded=web_snapshot['output'][key]
        current=web_recomputed['output'][key]
        print(f'{key:32s}  now={current:.9f}  saved={recorded:.9f}  delta={current-recorded:+.2e}')
        assert abs(current-recorded)<1e-5, f'Web solver value drift: {key}'


In [ ]:
# Actual web-solver dispersion curve and the two distinct derivatives
points=(web_recomputed or web_snapshot)['dispersion_curve']
diagnostics=(web_recomputed or web_snapshot)['output']
dd=pd.DataFrame(points).sort_values('wavelength_nm')
l0=diagnostics['lambda0_nm']
S0=diagnostics['slope_zero_ps_nm2_km']
S1550=diagnostics['slope_1550_ps_nm2_km']
D1550=diagnostics['D1550_ps_nm_km']
fig,ax=plt.subplots(figsize=(11.5,5.2))
ax.plot(dd.wavelength_nm,dd.D_ps_nm_km,'-',lw=2.6,label='Web FEM D(λ)')
ax.axhline(0,lw=1,ls='--',color='gray')
ax.axvline(l0,ls=':',lw=1.4,color='#1a9e8d')
ax.axvline(1550,ls=':',lw=1.4,color='#e79b40')
ax.scatter([l0,1550],[0,D1550],zorder=4,s=60)
loc=np.linspace(l0-45,l0+45,2)
near1550=np.linspace(1500,1600,2)
ax.plot(loc,S0*(loc-l0),'--',color='#1a9e8d',lw=2,label=f'S0={S0:.5f} @ {l0:.2f}nm')
ax.plot(near1550,D1550+S1550*(near1550-1550),'--',color='#e79b40',lw=2,label=f'S1550={S1550:.5f} @ 1550nm')
ax.annotate(f'λ0={l0:.2f} nm',xy=(l0,0),xytext=(l0-105,7),arrowprops={'arrowstyle':'->'})
ax.annotate(f'D1550={D1550:.3f}',xy=(1550,D1550),xytext=(1425,25),arrowprops={'arrowstyle':'->'})
ax.set(xlabel='Wavelength (nm)',ylabel='Dispersion D (ps/(nm km))',
       title='YOFC G.654.E — S0 and S1550 are derivatives at DIFFERENT wavelengths')
ax.grid(alpha=.25);ax.legend(loc='lower right')
plt.tight_layout();plt.show()
print(f'YOFC table: S1550 <= 0.070, while S0 and lambda0 are NOT supplied.')


In [ ]:
# Updated outputs from exact v0.2.1 web engine (or the saved genuine rerun snapshot)
replay = web_recomputed or web_snapshot
output_dir=Path('validation_outputs');output_dir.mkdir(exist_ok=True)
(output_dir/'web_solver_rerun.json').write_text(json.dumps(replay,ensure_ascii=False,indent=2),encoding='utf-8')
pd.DataFrame(replay['spectrum']).to_csv(output_dir/'spectrum_web.csv',index=False)
pd.DataFrame(replay['mesh']).to_csv(output_dir/'web_mesh_convergence.csv',index=False)
pd.DataFrame(replay['dispersion_curve']).to_csv(output_dir/'dispersion_curve_web.csv',index=False)
table.to_csv(output_dir/'comparison_with_slopes.csv',index=False)
print('Saved updated v0.2.1 JSON/CSV validation outputs to',output_dir)
